# Kaggle irrigation need prediction

Author: Ebelechukwu Umezude

Predict Low, Medium or High irrigation need using weighted histogram gradient boosting.
This notebook includes the outputs from the original run. The validation balanced
accuracy was 0.9728; this is not a confirmed Kaggle leaderboard score.

Download the competition train.csv and test.csv separately, place them next to
this notebook and run cells from top to bottom. The final cell writes submission.csv.
The competition datasets are not redistributed here.


#### 1. Loading and inspecting the files in the Python notebook

In [1]:
import pandas as pd

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training shape:", train.shape)
print("Test shape:", test.shape)

print(train["Irrigation_Need"].value_counts())

Training shape: (630000, 21)
Test shape: (270000, 20)


Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


#### 2. Separating Irrigation_Need as the target and excluding the id from the predictors.

In [2]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=["id", "Irrigation_Need"])
y = train["Irrigation_Need"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Validation:", X_valid.shape)

Training: (504000, 19)
Validation: (126000, 19)


#### 3. Preparing the features

In [3]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

categorical_columns = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_columns = [
    col for col in X_train.columns
    if col not in categorical_columns
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_columns
        ),
        ("numbers", "passthrough", numeric_columns)
    ]
)

# The transformer outputs categorical columns first.
categorical_mask = (
    [True] * len(categorical_columns)
    + [False] * len(numeric_columns)
)

print("Categorical columns:", categorical_columns)
print("Numeric columns:", numeric_columns)

Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numeric columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


#### 4. Train the model

In [4]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", HistGradientBoostingClassifier(
        categorical_features=categorical_mask,
        learning_rate=0.10,
        max_iter=200,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        early_stopping=True,
        random_state=42
    ))
])

weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

print("Training started...")

model.fit(
    X_train,
    y_train,
    classifier__sample_weight=weights
)

print("Training finished.")

Training started...
Training finished.


#### 5. Evaluate on validation data

In [5]:
valid_predictions = model.predict(X_valid)

# Baseline: always predict the most common training class.
majority_class = y_train.mode()[0]
baseline_predictions = np.full(len(y_valid), majority_class)

print(
    "Baseline balanced accuracy:",
    round(balanced_accuracy_score(
        y_valid, baseline_predictions
    ), 4)
)

print(
    "Model balanced accuracy:",
    round(balanced_accuracy_score(
        y_valid, valid_predictions
    ), 4)
)

print(
    "Model ordinary accuracy:",
    round(accuracy_score(
        y_valid, valid_predictions
    ), 4)
)

labels = ["Low", "Medium", "High"]

print("\nPerformance by class:")
print(classification_report(
    y_valid,
    valid_predictions,
    labels=labels,
    digits=4,
    zero_division=0
))

matrix = confusion_matrix(
    y_valid,
    valid_predictions,
    labels=labels
)

display(pd.DataFrame(
    matrix,
    index=["Actual Low", "Actual Medium", "Actual High"],
    columns=["Predicted Low", "Predicted Medium", "Predicted High"]
))

Baseline balanced accuracy: 0.3333
Model balanced accuracy: 0.9728
Model ordinary accuracy: 0.9842

Performance by class:
              precision    recall  f1-score   support

         Low     0.9869    0.9949    0.9909     73983
      Medium     0.9877    0.9703    0.9789     47815
        High     0.9004    0.9534    0.9261      4202

    accuracy                         0.9842    126000
   macro avg     0.9583    0.9728    0.9653    126000
weighted avg     0.9843    0.9842    0.9842    126000



,Predicted Low,Predicted Medium,Predicted High
Actual Low,73602,381,0
Actual Medium,977,46395,443
Actual High,0,196,4006


#### 6. Model Retraining with whole data

In [6]:
from sklearn.base import clone

# Create a fresh copy of the model
final_model = clone(model)

# Give each class balanced importance
full_weights = compute_sample_weight(
    class_weight="balanced",
    y=y
)

print("Training on all training rows...")

final_model.fit(
    X,
    y,
    classifier__sample_weight=full_weights
)

# Use exactly the same feature columns as during training
test_features = test[X.columns]
test_predictions = final_model.predict(test_features)

# Create the required Kaggle submission
submission = pd.DataFrame({
    "id": test["id"].to_numpy(),
    "Irrigation_Need": test_predictions
})

# Check the file before saving
assert len(submission) == len(test)
assert submission["id"].is_unique
assert not submission.isna().any().any()
assert set(submission["Irrigation_Need"]).issubset(
    {"Low", "Medium", "High"}
)

submission.to_csv("submission.csv", index=False)

print("Finished! Created submission.csv")
print("Submission shape:", submission.shape)
display(submission.head())
print(submission["Irrigation_Need"].value_counts())

Training on all training rows...
Finished! Created submission.csv
Submission shape: (270000, 2)


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


Irrigation_Need
Low       159734
Medium    100677
High        9589
Name: count, dtype: int64
